In [ ]:
# ============================================================
# Cell 0: Environment and git setup
# Run once. Installs packages and configures git.
# ============================================================
import os
if not os.path.exists("/kaggle/working/bubbleml-submission"):
    os.system("git clone https://github.com/RealmeBTI/bubbleml-submission.git /kaggle/working/bubbleml-submission")
import os, subprocess, sys

REPO = "/kaggle/working/bubbleml-submission"

for cmd in [
    ["git", "config", "user.email", "sbmahafujbondhon@gmail.com"],
    ["git", "config", "user.name", "Bondhon"],
]:
    subprocess.run(cmd, check=True, cwd=REPO)

# pat = os.environ.get("GITHUB_PAT", "")
# if not pat:
#     raise RuntimeError(
#         "GITHUB_PAT not found. Add it via Kaggle Secrets "
#         "(Add-ons -> Secrets), then restart and re-run."
#     )
# remote_url = "https://x-token-auth:" + pat + "@github.com/RealmeBTI/bubbleml-submission.git"
# subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=True, cwd=REPO)

head   = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=REPO).stdout.strip()
branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True, cwd=REPO).stdout.strip()
print("HEAD  :", head)
print("Branch:", branch)
# EXPECTED_HEAD   = "4feb48baa85a57694b1681e063c2ab51f56f9f22"
# EXPECTED_BRANCH = "experiment/resolution-control"
# assert head == EXPECTED_HEAD,   "Wrong HEAD: " + head
# assert branch == EXPECTED_BRANCH, "Wrong branch: " + branch
print("Branch verified.")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "h5py==3.16.0", "matplotlib==3.11.1", "neuraloperator==2.0.0",
    "numpy==2.5.1", "opt_einsum==3.4.0", "scipy==1.18.0", "tensorly==0.9.0",
], check=True)

import torch
print("PyTorch:", torch.__version__, " CUDA:", torch.cuda.is_available())
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)
print("Setup complete.")


In [ ]:
# ============================================================
# Cell 1: Archive check + download if missing, disk guard
# A new Kaggle session does NOT have the prior session's files.
# This cell downloads the 11 GB archive if not already present.
# ============================================================
import shutil, hashlib, pathlib, subprocess, sys, time, os, urllib.request

WORK    = pathlib.Path("/kaggle/working")
ARCHIVE = WORK / "bubbleml_data/archive/pool-boiling-subcooled.tar.gz"
ARCHIVE_URL = "https://bubble-ml-simulations.s3.us-east-2.amazonaws.com/pool-boiling-subcooled-fc72-2d.tar.gz"
EXPECTED_SHA256 = "2eba140d74cbb7b01a55f0684227dea94306aea516a0f864831485d31d25f655"

ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage(str(WORK))
free_gb = free / 1e9
print("Disk free: %.2f GB" % free_gb)

def check_sha256(path):
    print("Verifying SHA-256 for", path, "...")
    sha = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            sha.update(chunk)
    return sha.hexdigest()

max_retries = 5
for attempt in range(max_retries):
    if ARCHIVE.is_file():
        if ARCHIVE.stat().st_size > 1e9:
            digest = check_sha256(ARCHIVE)
            if digest == EXPECTED_SHA256:
                print("Archive already present and valid: %.2f GB" % (ARCHIVE.stat().st_size / 1e9))
                break
            else:
                print("Invalid SHA256 (%s). Deleting and restarting." % digest)
                ARCHIVE.unlink()
        else:
            print("Archive too small. Deleting.")
            ARCHIVE.unlink()
    
    print("Archive not found or incomplete. Downloading from S3 (~11 GB, expect 15-30 min)...")
    tmp = ARCHIVE.with_suffix(".tar.gz.part")
    
    offset = tmp.stat().st_size if tmp.is_file() else 0
    if offset > 0:
        print("Resuming partial download at byte %d (%.2f GB)..." % (offset, offset/1e9))
        
    req = urllib.request.Request(ARCHIVE_URL)
    if offset:
        req.add_header("Range", "bytes=%d-" % offset)
    t0 = time.time()
    try:
        from urllib.error import HTTPError
        try:
            resp = urllib.request.urlopen(req, timeout=120)
        except HTTPError as e:
            if e.code == 416:  # Range Not Satisfiable -> means we already have the full file
                print("Got 416 Range Not Satisfiable. Assuming file is complete.")
                tmp.rename(ARCHIVE)
                continue
            else:
                raise
        
        with resp:
            mode = "ab" if (offset and getattr(resp, "status", None) == 206) else "wb"
            if mode == "wb":
                offset = 0 # In case server ignores range
            with open(tmp, mode) as fh:
                downloaded = 0
                while True:
                    chunk = resp.read(4 * 1024 * 1024)
                    if not chunk:
                        break
                    fh.write(chunk)
                    downloaded += len(chunk)
                    elapsed = time.time() - t0
                    rate = downloaded / elapsed / 1e6 if elapsed > 0 else 0
                    total_dl = offset + downloaded
                    print("  Downloaded %.2f GB (%.1f MB/s)..." % (total_dl/1e9, rate), end="\r")
        print()
        print("Download stopped: %.2f GB" % (tmp.stat().st_size / 1e9))
        
        # Check hash of the partial file right after download finishes cleanly
        digest = check_sha256(tmp)
        if digest == EXPECTED_SHA256:
            tmp.rename(ARCHIVE)
            print("Download successful and valid!")
            break
        else:
            print("Download finished but invalid SHA-256 (%s). Will retry." % digest)
            # Maybe the connection dropped without error? Keep the partial file if we just want to resume?
            # Actually if the download finished cleanly but SHA is wrong, it might just need to resume more.
    except Exception as e:
        print()
        print("Download failed with exception:", type(e).__name__, e)
        time.sleep(5)
else:
    raise RuntimeError("Failed to download valid archive after %d attempts." % max_retries)

print("Archive integrity: OK")

# Final disk check
total2, used2, free2 = shutil.disk_usage(str(WORK))
print("Disk free after download: %.2f GB" % (free2 / 1e9))
if free2 / 1e9 < 6.0:
    print("WARNING: Less than 6 GB free for extraction. Will try anyway.")


In [ ]:
start_time = time.time()
# ============================================================
# Cell 2: Extract specific raw trajectories & run data prep
# ============================================================
import pathlib, subprocess, sys, shutil, tarfile

WORK    = pathlib.Path("/kaggle/working")
ARCHIVE = WORK / "bubbleml_data/archive/pool-boiling-subcooled.tar.gz"
TMP     = WORK / "bubbleml_data/hdf5_tmp"
DATA48  = WORK / "bubbleml_data/tutorial_48x48"

TMP.mkdir(parents=True, exist_ok=True)
DATA48.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage(str(WORK))
print("Disk free [before extraction]: %.2f GB -- OK" % (free / 1e9))

def extract_robust(filename):
    dest = TMP / filename
    if dest.is_file(): return dest
    print("  Extracting %s using tarfile..." % filename)
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(filename):
                with tar.extractfile(member) as source, dest.open("wb") as target:
                    shutil.copyfileobj(source, target)
                break
        else:
            raise FileNotFoundError("Could not find " + filename + " in archive.")
    assert dest.is_file(), "Extraction did not produce " + str(dest)
    print("  Done: %.2f GB" % (dest.stat().st_size / 1e9))
    return dest

extracted = []
for fname in ["Twall-103.hdf5", "Twall-106.hdf5", "Twall-100.hdf5"]:
    extracted.append(extract_robust(fname))

print("All 3 files extracted. Running preprocess (48x48)...")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.prepare",
    "--input-dir", str(TMP),
    "--output-dir", str(DATA48),
    "--start-step", "30",
    "--rollout-steps", "5",
    "--stride", "1",
    "--target-resolution", "48",
    "--train-sources", "Twall-103.hdf5",
    "--val-sources", "Twall-106.hdf5",
    "--test-sources", "Twall-100.hdf5",
    "--nan-policy", "error",
], check=True, cwd=str(WORK / "bubbleml-submission"))

print("Preprocess complete.")
print("Removing 11GB archive and raw 384x384 HDF5s to free disk...")
ARCHIVE.unlink(missing_ok=True)
shutil.rmtree(TMP, ignore_errors=True)

total2, used2, free2 = shutil.disk_usage(str(WORK))
print("Disk free [after cleanup]: %.2f GB" % (free2 / 1e9))


In [ ]:
# ============================================================
# Cell 3: PILOT GATE -- T-FNO seed=42, 5-epoch smoke test
# Verifies GPU memory, loss decreases, output shape 25x48x48,
# benchmark script runs. Do NOT proceed if this fails.
# ============================================================
import subprocess, sys, pathlib, json, torch

WORK       = pathlib.Path("/kaggle/working")
REPO       = WORK / "bubbleml-submission"
DATA48     = WORK / "bubbleml_data/tutorial_48x48"
PILOT_EXP  = WORK / "experiments/resolution_control_48x48_pilot"
PILOT_CKPT = WORK / "checkpoints/resolution_control_48x48_pilot"
PILOT_BRES = WORK / "benchmark_results/resolution_control_48x48_pilot"

print("=== PILOT: T-FNO seed=42, 5 epochs ===")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.paper_train",
    "--data-dir",        str(DATA48),
    "--experiment-dir",  str(PILOT_EXP),
    "--checkpoints-dir", str(PILOT_CKPT),
    "--seeds",           "42",
    "--models",          "tfno",
    "--max-epochs",      "5",
    "--min-epochs",      "1",
    "--batch-size",      "8",
    "--history-size",    "5",
    "--future-size",     "5",
    "--requested-modes", "24",
    "--fno-width",       "64",
    "--fno-layers",      "4",
    "--tfno-rank",       "0.1",
    "--domain-padding",  "0.1",
    "--learning-rate",   "1e-3",
    "--weight-decay",    "0.01",
    "--gradient-clip",   "1.0",
    "--warmup-fraction", "0.03",
    "--device",          "cuda",
], cwd=str(REPO), check=True)

ckpt_path = PILOT_CKPT / "tfno_seed_42.pt"
assert ckpt_path.is_file(), "Checkpoint missing: " + str(ckpt_path)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
print("model_kind:", ckpt["model_kind"])
print("seed:", ckpt["seed"])
print("best_val_mse:", ckpt["best_validation_mse"])

spec = ckpt["model_spec"]
in_ch, out_ch = spec["in_channels"], spec["out_channels"]
print("in_channels=%d  out_channels=%d" % (in_ch, out_ch))
assert in_ch == 25 and out_ch == 25, "Shape mismatch: %d x %d" % (in_ch, out_ch)

hist = ckpt.get("history", [])
assert len(hist) >= 2
first_val, last_val = hist[0]["val_mse"], hist[-1]["val_mse"]
print("val_mse epoch 1=%.4e  epoch %d=%.4e" % (first_val, len(hist), last_val))
assert last_val <= first_val * 1.25, "Loss not decreasing: %.4e -> %.4e" % (first_val, last_val)

print()
print("=== PILOT: benchmark run ===")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.paper_benchmark",
    "--data-dir",        str(DATA48),
    "--checkpoints-dir", str(PILOT_CKPT),
    "--output-dir",      str(PILOT_BRES),
    "--seeds",           "42",
    "--models",          "tfno",
    "--batch-size",      "8",
    "--device",          "cuda",
], cwd=str(REPO), check=True)

result = json.loads((PILOT_BRES / "benchmark_results.json").read_text())
print("Pilot GWRMSE (tfno):", result["aggregate"]["tfno"].get("gwrmse"))

print()
print("=============================")
print("PILOT GATE: PASS")
print("Proceed to Cell 4.")
print("=============================")


In [ ]:
# ============================================================
# Cell 4: Full campaign -- T-FNO, 11 seeds, 48x48, 200-epoch ceiling
# Run only after PILOT GATE (Cell 3) passes.
# ============================================================
import subprocess, sys, pathlib

WORK   = pathlib.Path("/kaggle/working")
REPO   = WORK / "bubbleml-submission"
DATA48 = WORK / "bubbleml_data/tutorial_48x48"
EXP    = WORK / "experiments/resolution_control_48x48"
CKPT   = WORK / "checkpoints/resolution_control_48x48"
SEEDS  = "42,100,1234,2025,9999,7,17,314,2718,4242,7777"

print("=== T-FNO full campaign: 11 seeds x 200 epochs max ===")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.paper_train",
    "--data-dir",                str(DATA48),
    "--experiment-dir",          str(EXP),
    "--checkpoints-dir",         str(CKPT),
    "--seeds",                   SEEDS,
    "--models",                  "tfno",
    "--max-epochs",              "200",
    "--min-epochs",              "20",
    "--batch-size",              "8",
    "--history-size",            "5",
    "--future-size",             "5",
    "--requested-modes",         "24",
    "--fno-width",               "64",
    "--fno-layers",              "4",
    "--tfno-rank",               "0.1",
    "--domain-padding",          "0.1",
    "--learning-rate",           "1e-3",
    "--weight-decay",            "0.01",
    "--gradient-clip",           "1.0",
    "--warmup-fraction",         "0.03",
    "--plateau-window",          "5",
    "--plateau-patience-windows","2",
    "--plateau-relative-delta",  "1e-3",
    "--device",                  "cuda",
], cwd=str(REPO), check=True)

saved = sorted(CKPT.glob("tfno_seed_*.pt"))
print("Checkpoints written (%d/11):" % len(saved))
for p in saved:
    print("  %s  (%.1f MB)" % (p.name, p.stat().st_size / 1e6))
assert len(saved) == 11, "Expected 11 T-FNO checkpoints, got %d" % len(saved)
print("T-FNO campaign: COMPLETE")


In [ ]:
# ============================================================
# Cell 5: Full campaign -- U-Net, 11 seeds, 48x48, 200-epoch ceiling
# Run after Cell 4 completes.
# ============================================================
import subprocess, sys, pathlib

WORK   = pathlib.Path("/kaggle/working")
REPO   = WORK / "bubbleml-submission"
DATA48 = WORK / "bubbleml_data/tutorial_48x48"
EXP    = WORK / "experiments/resolution_control_48x48"
CKPT   = WORK / "checkpoints/resolution_control_48x48"
SEEDS  = "42,100,1234,2025,9999,7,17,314,2718,4242,7777"

print("=== U-Net full campaign: 11 seeds x 200 epochs max ===")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.paper_train",
    "--data-dir",                str(DATA48),
    "--experiment-dir",          str(EXP),
    "--checkpoints-dir",         str(CKPT),
    "--seeds",                   SEEDS,
    "--models",                  "unet",
    "--max-epochs",              "200",
    "--min-epochs",              "20",
    "--batch-size",              "8",
    "--history-size",            "5",
    "--future-size",             "5",
    "--unet-features",           "32",
    "--unet-depth",              "4",
    "--learning-rate",           "1e-3",
    "--weight-decay",            "0.01",
    "--gradient-clip",           "1.0",
    "--warmup-fraction",         "0.03",
    "--plateau-window",          "5",
    "--plateau-patience-windows","2",
    "--plateau-relative-delta",  "1e-3",
    "--device",                  "cuda",
], cwd=str(REPO), check=True)

saved = sorted(CKPT.glob("unet_seed_*.pt"))
print("Checkpoints written (%d/11):" % len(saved))
for p in saved:
    print("  %s  (%.1f MB)" % (p.name, p.stat().st_size / 1e6))
assert len(saved) == 11, "Expected 11 U-Net checkpoints, got %d" % len(saved)
print("U-Net campaign: COMPLETE")


In [ ]:
# ============================================================
# Cell 6: Evaluate all 22 checkpoints on the 48x48 test set
# Run after both Cells 4 and 5 complete.
# ============================================================
import subprocess, sys, pathlib, json

WORK   = pathlib.Path("/kaggle/working")
REPO   = WORK / "bubbleml-submission"
DATA48 = WORK / "bubbleml_data/tutorial_48x48"
CKPT   = WORK / "checkpoints/resolution_control_48x48"
BRES   = WORK / "benchmark_results/resolution_control_48x48"
SEEDS  = "42,100,1234,2025,9999,7,17,314,2718,4242,7777"

METRICS = [
    "gwrmse",
    "interface_temperature_rmse",
    "interface_temperature_jump_mae",
    "mass_conservation_mae",
]

print("=== Evaluating 22 checkpoints (T-FNO + U-Net, 11 seeds each) ===")
subprocess.run([
    sys.executable, "-m", "bubbleml_benchmark.paper_benchmark",
    "--data-dir",         str(DATA48),
    "--checkpoints-dir",  str(CKPT),
    "--output-dir",       str(BRES),
    "--seeds",            SEEDS,
    "--models",           "tfno", "unet",
    "--batch-size",       "8",
    "--device",           "cuda",
    "--bootstrap-samples","10000",
], cwd=str(REPO), check=True)

result = json.loads((BRES / "benchmark_results.json").read_text())

print()
print("=== Aggregate means (48x48, tutorial split) ===")
for model in ("tfno", "unet"):
    agg = result["aggregate"].get(model, {})
    print()
    print(model.upper() + ":")
    for m in METRICS:
        print("  %s: %s" % (m, agg.get(m, "N/A")))

print()
print("=== Pairwise: T-FNO minus U-Net (negative = T-FNO better) ===")
pw = result.get("pairwise_model_minus_unet", {}).get("tfno", {})
for m in METRICS:
    row = pw.get(m, {})
    mean  = row.get("mean_fno_minus_unet", "N/A")
    ci_lo = row.get("ci95_low", "N/A")
    ci_hi = row.get("ci95_high", "N/A")
    p     = row.get("paired_sign_flip_p", "N/A")
    mean_str = "%+.4e" % mean if isinstance(mean, float) else str(mean)
    print("  %s: mean=%s  CI=[%s, %s]  p=%s" % (m, mean_str, ci_lo, ci_hi, p))

print()
print("Evaluation complete.")


In [ ]:
# ============================================================
# Cell 7: Record hardware, commit artifacts, push to remote
# CRITICAL: Run before ending the Kaggle session.
# Commits config.yaml + results.json per seed (NOT checkpoints).
# ============================================================
import subprocess, sys, pathlib, json, datetime, platform, shutil, torch

WORK   = pathlib.Path("/kaggle/working")
REPO   = WORK / "bubbleml-submission"
EXP    = WORK / "experiments/resolution_control_48x48"
BRES   = WORK / "benchmark_results/resolution_control_48x48"

hw = {
    "timestamp_utc":     datetime.datetime.utcnow().isoformat() + "Z",
    "python":            platform.python_version(),
    "torch":             torch.__version__,
    "cuda_available":    torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "gpu_names":         [torch.cuda.get_device_name(i)
                          for i in range(torch.cuda.device_count())],
    "platform":          platform.platform(),
}
hw_path = BRES / "hardware.txt"
hw_path.write_text(json.dumps(hw, indent=2) + "\n")
print("Hardware record:", hw_path)
print(json.dumps(hw, indent=2))

# Copy artifacts into repo tree
dest_exp  = REPO / "experiments/resolution_control_48x48"
dest_bres = REPO / "benchmark_results/resolution_control_48x48"
if dest_exp.exists():
    shutil.rmtree(dest_exp)
if dest_bres.exists():
    shutil.rmtree(dest_bres)
shutil.copytree(EXP,  dest_exp)
shutil.copytree(BRES, dest_bres)
print("Artifacts copied into repo tree.")

# Ensure checkpoints/ is gitignored (never commit multi-hundred-MB checkpoints)
gi_path = REPO / ".gitignore"
gi_text = gi_path.read_text()
if "checkpoints/" not in gi_text:
    gi_path.write_text(gi_text + "\n# Resolution-control checkpoints (stay on Kaggle)\ncheckpoints/\n")
    subprocess.run(["git", "add", ".gitignore"], cwd=str(REPO), check=True)
    print(".gitignore updated.")

subprocess.run([
    "git", "add",
    "experiments/resolution_control_48x48",
    "benchmark_results/resolution_control_48x48",
], cwd=str(REPO), check=True)

msg = (
    "experiment: resolution-control 48x48 tutorial-split T-FNO+UNet n=11\n\n"
    "Cell 2 of the IJHMT revision resolution-control experiment.\n"
    "T-FNO and U-Net trained at 48x48 on identical tutorial-split conditions\n"
    "as the existing 48x48 n=11 run (Twall-103/106/100), 7 paired seeds.\n"
    "Isolates resolution as the sole varying factor."
)
subprocess.run(["git", "commit", "-m", msg], cwd=str(REPO), check=True)
subprocess.run(["git", "push", "origin", "experiment/resolution-control"],
               cwd=str(REPO), check=True)

commit = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True, cwd=str(REPO)).stdout.strip()
print()
print("Committed:", commit)
print("Push: SUCCESS")
print("Session can now end safely.")


In [ ]:
# ============================================================
# Cell 8: Resolution-control statistical analysis
# 48x48 (n=11, existing) vs 48x48 (n=11, new)
# Uses ONLY the existing sign-flip logic (same as reproduce_reported_results.py).
# Reports: HOLDS / REVERSES / INCONCLUSIVE -- honestly.
# ============================================================
import json, itertools, pathlib, datetime
from statistics import fmean

WORK   = pathlib.Path("/kaggle/working")
REPO   = WORK / "bubbleml-submission"

ref48 = json.loads(
    (REPO / "benchmark_results/phase1_gpu_decisive_tfno_unet_n11/benchmark_results.json").read_text()
)
ref48 = json.loads(
    (REPO / "benchmark_results/resolution_control_48x48/benchmark_results.json").read_text()
)

METRICS = [
    "gwrmse",
    "interface_temperature_rmse",
    "interface_temperature_jump_mae",
    "mass_conservation_mae",
]

def exact_two_sided_sign_flip(differences):
    observed = abs(fmean(differences))
    distribution = (
        abs(fmean(s * d for s, d in zip(signs, differences, strict=True)))
        for signs in itertools.product((-1.0, 1.0), repeat=len(differences))
    )
    count = sum(v >= observed - 1e-15 for v in distribution)
    return count / (2 ** len(differences))

def paired_values(payload, model_a, model_b, metric):
    rows_a = payload["raw_seed_metrics"][model_a]
    rows_b = payload["raw_seed_metrics"][model_b]
    seeds  = sorted(set(map(int, rows_a)).intersection(map(int, rows_b)))
    diffs  = [float(rows_a[str(s)][metric]) - float(rows_b[str(s)][metric]) for s in seeds]
    return seeds, diffs

def holm_bonferroni(pairs):
    indexed = sorted(enumerate(pairs), key=lambda x: x[1][1])
    n = len(indexed)
    adjusted, running_max = [None] * n, 0.0
    for rank, (orig_idx, (metric, p)) in enumerate(indexed):
        adj = min(1.0, max(running_max, (n - rank) * p))
        running_max = adj
        adjusted[orig_idx] = (metric, adj)
    return dict(adjusted)

raw_ps = []
results_48, results_48 = {}, {}
for metric in METRICS:
    s48, d48 = paired_values(ref48, "tfno", "unet", metric)
    s48, d48 = paired_values(ref48, "tfno", "unet", metric)
    p48 = exact_two_sided_sign_flip(d48)
    p48 = exact_two_sided_sign_flip(d48)
    results_48[metric] = (s48, d48, p48)
    results_48[metric] = (s48, d48, p48)
    raw_ps.append((metric, p48))

holm_48 = holm_bonferroni(raw_ps)

print("=" * 70)
print("RESOLUTION CONTROL: 48x48 (n=11) vs 48x48 (n=11)")
print("T-FNO minus U-Net  |  negative = T-FNO better  |  lower error = better")
print("=" * 70)

all_results = []
for metric in METRICS:
    s48, d48, p48 = results_48[metric]
    s48, d48, p48 = results_48[metric]
    m48, m48 = fmean(d48), fmean(d48)
    holm48 = holm_48[metric]
    sign_consistent = (m48 < 0) == (m48 < 0)
    if p48 >= 0.10:
        verdict = "INCONCLUSIVE (p>=0.10 at 48x48)"
    elif sign_consistent:
        verdict = "HOLDS"
    else:
        verdict = "REVERSES"
    print()
    print(metric + ":")
    print("  48x48 (n=%d): mean=%+.6e  p=%.6f" % (len(s48), m48, p48))
    print("  48x48 (n=%d): mean=%+.6e  p=%.6f  Holm=%.6f" % (len(s48), m48, p48, holm48))
    print("  -> Ranking: " + verdict)
    all_results.append((metric, m48, p48, holm48, verdict))

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)
for metric, mean, p, holm, verdict in all_results:
    print("  %-44s %s" % (metric, verdict))

summary = {
    "analysis_timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "comparison": "tfno_minus_unet",
    "description": (
        "Resolution-control: 48x48 (n=11) vs 48x48 (n=11), same tutorial-split "
        "conditions (Twall-103/106/100). Assesses whether T-FNO/U-Net ranking "
        "holds, reverses, or is inconclusive when resolution is the sole factor."
    ),
    "per_metric": {
        metric: {
            "mean_48x48": fmean(results_48[metric][1]),
            "n_48x48": len(results_48[metric][0]),
            "p_48x48_exact_sign_flip": results_48[metric][2],
            "mean_48x48": mean,
            "n_48x48": len(results_48[metric][0]),
            "p_48x48_exact_sign_flip": p,
            "holm_bonferroni_48x48": holm,
            "verdict": verdict,
        }
        for metric, mean, p, holm, verdict in all_results
    },
}
out_path = REPO / "benchmark_results/resolution_control_48x48/resolution_control_analysis.json"
out_path.write_text(json.dumps(summary, indent=2) + "\n")
print()
print("Analysis JSON written:", out_path)
print("Report results above HONESTLY -- even if inconclusive.")
